<a href="https://colab.research.google.com/github/oliviadellaglio/ds2002-fa26/blob/main/2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [28]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one: line item of one order

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [29]:
import pandas as pd

#flatten nested orders into one row per line item
orders = pd.json_normalize(
    orders_json, #list to flatten
    record_path='lines', #one row per line item
    meta=["order", "vendor_id"], #copy fields onto every row from that order
)[["order", "vendor_id", "item", "qty"]] #put key columns first

#check that there are 4 rows and a quantity of 7
assert len(orders) == 4 and orders["qty"].sum() == 7
orders


,order,vendor_id,item,qty
0,1,V-01,Cheeseburger,2
1,1,V-01,Soda,1
2,2,V-10,Foam Finger,1
3,3,V-07,Hot Dog,3


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [30]:
vendors = pd.DataFrame(vendors_json)

joined = orders.merge(
    vendors,
    on="vendor_id", #join on vendor_id
    how="left", #keeps every line item
    indicator=True, #adds _merge column to show match status
    validate="many_to_one", #errors if vendor_id is duplicated in vendors
)

#makes sure join has not add or dropped rows
assert len(joined)==4 and joined["qty"].sum() == 7

#print rows with no vendor match
unmatched = joined[joined["_merge"]=="left_only"]
print(unmatched["vendor_id"].unique())
print(unmatched["qty"].sum())

['V-07']
3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [31]:
joined["vendor_label"] = joined["name"].fillna("UNMATCHED (" + joined["vendor_id"] + ")")
#replace NaN with "UNMATCHED" and vendor_id

qty_by_vendor = (
    joined.groupby("vendor_label", as_index=False)["qty"].sum()
    .sort_values(["qty", "vendor_label"], ascending=[False, True])
    .reset_index(drop=True)
)

#make sure no rows are added or dropped
assert qty_by_vendor["qty"].sum() == 7

qty_by_vendor

#unmatched vendor gets UNMATCHED label instead of NaN.
#The name must be changed because groupby drops NaN keys, so the order would not be
#visable on the table. vendor_id still included so the vendor can be identified.
#This ensures the rows still equal 7

,vendor_label,qty
0,Hoos Burgers,3
1,UNMATCHED (V-07),3
2,Cav Merch,1


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [32]:
UNKNOWN = 'UNKNOWN (unmatched vendor)' #placeholder for rows with NaN for vendor


#copy row's zone into new column and replace NaN with UNKNOWN so groupby does not drop
joined['zone_label'] = joined['zone'].fillna("UNKNOWN (unmatched vendor)")

qty_by_zone = (joined.groupby('zone_label', as_index=False)["qty"].sum()
.sort_values(['qty', 'zone_label'], ascending=[False, True])
.reset_index(drop=True)
)

#keep rows without placeholders
known = qty_by_zone[qty_by_zone["zone_label"] != UNKNOWN]

#count number of units without UNKNOWN
known_total = known['qty'].sum()


print(known_total)

#add unknown and known quanities to ensure no rows are dropped
assert qty_by_zone["qty"].sum() == 7

qty_by_zone

4


,zone_label,qty
0,A,4
1,UNKNOWN (unmatched vendor),3


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [33]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

rows = [] #becomes table rows
no_lines = [] #hold order numbers with no lines

#for each order:
for o in ragged_json :
  lines = o.get("lines") #if lines is missing, returns none and moves to if
  if not lines:
    no_lines.append(o["order"]) #remember order number
    continue #skip to next order

  #for each item this order add to rows
  for line in lines:
    rows.append({
        "order": o["order"],
        "vendor_id": o["vendor_id"],
        "item": line.get("item"),
        "qty": line.get("qty"),
    })

#turn list of rows into a table with these columns in order
ragged = pd.DataFrame(rows, columns=["order", "vendor_id", "item", "qty"])

#Use number tyoe that can be empty if qty is none without getting confused for 0
ragged["qty"] = ragged["qty"].astype("Int64")

#add a true/false column to data frame to show if quanity is missing
ragged["qty_missing"] = ragged["qty"].isna()

#print orders with no rows
print("Orders with no lines (no rows produced):", no_lines)

ragged

Orders with no lines (no rows produced): [5]


,order,vendor_id,item,qty,qty_missing
0,4,V-01,Soda,2,False
1,6,V-01,Fries,<NA>,True


### Q6 — Validate

In [34]:
assert len(orders) == 4
assert orders['qty'].sum() == 7
assert len(joined) == len(orders), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

A) One row per line item is useful beacause the tables are easy to read and count up. if there was one row per order, then the items get put into a list inside that row, which is harder to sort. You can easly determine how many units a vebndor sold, which zone sold the most, and how many of an item were sold.

B) A missing quanity means that the line item exists, but we do not know the quantity. Zero means the item exists, but none were sold (there is no question). If you mix them up, then reported quanities can be wrong, not telling the user that there was another sale, but quantity is unaccounted for. Averages will also be wrong if NaN is replaced with 0. Using Int64 keeps missing values as NA and keeps them seperate from 0. The qty_missing column also helps because you can filter on it.